In [8]:
from copy import copy
from typing import List, Tuple
from dataclasses import dataclass

import fitz
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, precision_score, recall_score

from dataUtils import pages_with_annotations
from transformers import BertModel, BertConfig
from transformers import AutoModel, AutoTokenizer

In [17]:
VAL_PDF_DOC_PATH = "C:\\main\\GitHub\\documentReviewSystem\\NERTraining\\TestData\\classic_machine_learning.pdf"
VAL_PDF_DOC_PATH

'C:\\main\\GitHub\\documentReviewSystem\\NERTraining\\TestData\\classic_machine_learning.pdf'

In [9]:
EMBEDDING_MODEL_NAME = 'Snowflake/snowflake-arctic-embed-l-v2.0'
TOKENIZER = AutoTokenizer.from_pretrained(EMBEDDING_MODEL_NAME)
EMBD_MODEL = AutoModel.from_pretrained(EMBEDDING_MODEL_NAME)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3281.71it/s]


In [14]:
class ModelInference:
    def __init__(self, pdf_path: str, pdf_page: int, tokenizer: AutoTokenizer):
        self.pdf_path = pdf_path
        self.pdf_page = pdf_page
        self.tokenizer = tokenizer
    
    def get_pdf_text(self) -> str:
        with fitz.open(self.pdf_path) as val_doc:
            page = val_doc[self.pdf_page]
            page_text = page.get_text()
        return page_text

    def infer(self, model: nn.Module, device: str = 'cuda') -> List[str]:
        tokenized_text = TOKENIZER(self.get_pdf_text()).input_ids
        tensor_text = torch.tensor(tokenized_text).unsqueeze(dim = 0).to(device = device)
        prediction = model(tensor_text).squeeze(dim = 0)
        label_prediction = prediction.softmax(dim = 1).argmax(dim = 1).cpu()
        inds_of_2 = (label_prediction == 2).nonzero(as_tuple = True)[0].tolist()
        # Collecting technologies to token_words_list
        token_words_list: List[List[int]] = []
        for single_2ind in inds_of_2:
            current_token_word = []
            if single_2ind == len(label_prediction) - 1:
                break
            current_ind = single_2ind + 1
            current_tag = label_prediction[current_ind]
            current_token_word.append(tokenized_text[current_ind - 1])
            while current_tag == 1:
                word_token = tokenized_text[current_ind]
                current_token_word.append(word_token)
                current_ind += 1
                if current_ind == len(label_prediction):
                    break
                else:
                    current_tag = label_prediction[current_ind]
            if len(current_token_word) not in [0, 1]:
                token_words_list.append(copy(current_token_word))
            current_token_word.clear()
        return self.tokenizer.decode(token_words_list, skip_special_tokens = True)

### Elman RNN model inference

In [18]:
# Different types of RNNs, biRNNs, and multi-layered RNNs 
class NERModelElman(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int,
                 embedding_model: AutoTokenizer, num_layers: int = 2, num_tags: int = 3, bi: bool = False):
        super().__init__()
        self.embedding_model = embedding_model
        self.classifier = nn.Linear(hidden_dim, num_tags)
        self.rnn = nn.RNN(input_size = input_dim, hidden_size = hidden_dim,
                          num_layers = num_layers, bidirectional = bi)

    def forward(self, x: torch.tensor) -> torch.tensor:
        """x is tensor of shape BxS where B is batch size and S is sequence length of inputs"""
        with torch.no_grad():
            emb_x = self.embedding_model(x).last_hidden_state
        emb_x = torch.permute(emb_x, (1, 0, 2)) 
        out, _ = self.rnn(emb_x)
        out = torch.permute(out, (1, 0, 2))
        y = self.classifier(out) # B x S x E
        return y

In [142]:
ner_model = NERModelElman(input_dim = 1024, hidden_dim = 256, 
                          num_layers = 2, embedding_model = EMBD_MODEL).to(device='cuda')

In [ ]:
# 2 layers, 256 hidden size
ner_model.rnn.load_state_dict(torch.load("C:\\main\\GitHub\\documentReviewSystem\\NERTraining\\models\\ELMAN_RNN_LAYERS_2_HIDDEN_256_epoch#55.pth"))

<All keys matched successfully>

In [157]:
model_inf = ModelInference(VAL_PDF_DOC_PATH, 5, TOKENIZER)

In [161]:
inf_result = model_inf.infer(ner_model, device = 'cuda')
inf_result

['данных. Очистка и преобразование данных - удаление лишних признаков, удаление непоказательных объектов, заполнение отсутствующих ',
 'значений, создание суррогатных признаков, преобразование шкал, воспроизводимость преобразования',
 'данных.']

### LSTM model inference

In [173]:
class NERModelLSTM(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int,
                 embedding_model: AutoTokenizer, num_layers: int = 2, num_tags: int = 3):
        super().__init__()
        self.embedding_model = embedding_model
        self.classifier = nn.Linear(hidden_dim, num_tags)
        self.rnn = nn.LSTM(input_size = input_dim, hidden_size = hidden_dim,
                           num_layers = num_layers, bidirectional = False, batch_first = False)

    def forward(self, x: torch.tensor) -> torch.tensor:
        """x is tensor of shape BxS where B is batch size and S is sequence length of inputs"""
        with torch.no_grad():
            emb_x = self.embedding_model(x).last_hidden_state
        emb_x = torch.permute(emb_x, (1, 0, 2)) 
        out, _ = self.rnn(emb_x)
        out = torch.permute(out, (1, 0, 2))
        y = self.classifier(out) # B x S x E
        return y

In [ ]:
ner_model = NERModelLSTM(input_dim = 1024, hidden_dim = 256,
                          num_layers = 1, embedding_model = EMBD_MODEL).to(device='cuda')

In [ ]:
# 1 layer 256 hidden size
ner_model.rnn.load_state_dict(torch.load("C:\\main\\GitHub\\documentReviewSystem\\NERTraining\\models\\LSTM_LAYERS_2_HIDDEN_256_epoch#98.pth"))

<All keys matched successfully>

In [ ]:
model_inf = ModelInference(VAL_PDF_DOC_PATH, 5, TOKENIZER)

In [181]:
inf_result = model_inf.infer(ner_model, device = 'cuda')
inf_result

['- MAE',
 'дообу',
 'чение и',
 'переоб',
 'чение.',
 'ance. Оценка',
 'набор, кривые',
 'обучения.',
 'учением.',
 'изация.',
 'валид',
 'параметр',
 'тке, ',
 'валидационный',
 'набор.',
 'варительный',
 'данных,',
 'данных.',
 'ательный (',
 'варительный)',
 'алы и',
 'ализация,',
 'ности,',
 'й,',
 'й в',
 '. О',
 'ка и',
 'данных -',
 'удаление',
 'лишних',
 'ков,',
 'объектов, заполнение от',
 'ствующих ',
 'значений, создание',
 'ков,',
 'кал, вос',
 'производимость']

### GRU model inference

In [186]:
class NERModelGRU(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int,
                 embedding_model: AutoTokenizer, num_layers: int = 2, num_tags: int = 3):
        super().__init__()
        self.embedding_model = embedding_model
        self.classifier = nn.Linear(hidden_dim, num_tags)
        self.rnn = nn.GRU(input_size = input_dim, hidden_size = hidden_dim,
                           num_layers = num_layers, bidirectional = False, batch_first = False)

    def forward(self, x: torch.tensor) -> torch.tensor:
        """x is tensor of shape BxS where B is batch size and S is sequence length of inputs"""
        with torch.no_grad():
            emb_x = self.embedding_model(x).last_hidden_state
        emb_x = torch.permute(emb_x, (1, 0, 2))
        out, _ = self.rnn(emb_x)
        out = torch.permute(out, (1, 0, 2))
        y = self.classifier(out) # B x S x E
        return y

In [192]:
ner_model = NERModelGRU(input_dim = 1024, hidden_dim = 256, 
                         num_layers = 1, embedding_model = EMBD_MODEL).to(device='cuda')

In [217]:
# 1 layer 256 hidden size
ner_model.rnn.load_state_dict(torch.load("C:\\main\\GitHub\\documentReviewSystem\\NERTraining\\models\\GRU_LAYERS_2_HIDDEN_256_epoch#67.pth"))

<All keys matched successfully>

In [221]:
model_inf = ModelInference(VAL_PDF_DOC_PATH, 5, TOKENIZER)

In [222]:
inf_result = model_inf.infer(ner_model, device = 'cuda')
inf_result

['. Не',
 'ценка сложности моделей',
 'способность моделей',
 '. Метод',
 'ы борьбы с недо',
 '. Регуляр',
 '. За',
 'дача выбора модели',
 'ы моделей',
 ', ',
 '. Тема 6. Предвар',
 'для моделей',
 'с учителем - реля',
 'ная форма данных',
 'ительный) анализ',
 'EDA) - анализ репрезентативности',
 'и типы, визуализация',
 '.']

### BERT MODEL inference

In [57]:
@dataclass(frozen=True)
class TrainConfig:
    BATCH_SIZE: int = 6
    LR: float = 1e-5
    BETAS: Tuple =  (0.9, 0.98)
    BIDIRECTIONAL: bool = False
train_config = TrainConfig()

In [58]:
class NERModelBERT(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int,
                 embedding_model: AutoTokenizer, num_layers: int = 2, num_tags: int = 3):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.embedding_model = embedding_model
        self.classifier = nn.Linear(input_dim, num_tags)
        self.rnn = BertModel(self.config).encoder

    @property
    def config(self):
        config = BertConfig(
            vocab_size=len(TOKENIZER.get_vocab()),
            hidden_size=self.input_dim,
            intermediate_size = self.hidden_dim,
            num_hidden_layers=1,
            num_attention_heads=4)
        return config

    def forward(self, x: torch.tensor) -> torch.tensor:
        """x is tensor of shape BxS where B is batch size and S is sequence length of inputs"""
        with torch.no_grad():
            emb_x = self.embedding_model(x).last_hidden_state
        emb_x = torch.permute(emb_x, (1, 0, 2))
        last_hidden_state = self.rnn(emb_x).last_hidden_state
        last_hidden_state = torch.permute(last_hidden_state, (1, 0, 2))
        y = self.classifier(last_hidden_state) # B x S x E
        return y

In [59]:
ner_model = NERModelBERT(input_dim = 1024, hidden_dim = 256, 
                         num_layers = 2, embedding_model = EMBD_MODEL).to(device='cuda')

In [60]:
ner_model.rnn.load_state_dict(torch.load("C:\\main\\GitHub\\documentReviewSystem\\NERTraining\\models\\BERT_MODEL_epoch#61.pth"))

<All keys matched successfully>

In [61]:
model_inf = ModelInference(VAL_PDF_DOC_PATH, 17, TOKENIZER)

In [70]:
inf_result = model_inf.infer(ner_model, device = 'cuda')
inf_result

['математ',
 'помощи инструмента',
 'выводы',
 'размерность',
 'зуал',
 'представляет собой',
 '3. Вы',
 'моделей.',
 'вариант предполагает',
 'Необходимо после',
 'та к']

### Elman RNN model inference

In [71]:
# Different types of RNNs, biRNNs, and multi-layered RNNs 
class NERModelElman(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int,
                 embedding_model: AutoTokenizer, num_layers: int = 2, num_tags: int = 3, bi: bool = False):
        super().__init__()
        self.embedding_model = embedding_model
        self.classifier = nn.Linear(2*hidden_dim if bi else hidden_dim, num_tags)
        self.rnn = nn.RNN(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers, bidirectional=bi)

    def forward(self, x: torch.tensor) -> torch.tensor:
        """x is tensor of shape BxS where B is batch size and S is sequence length of inputs"""
        with torch.no_grad():
            emb_x = self.embedding_model(x).last_hidden_state
        emb_x = torch.permute(emb_x, (1, 0, 2)) 
        out, _ = self.rnn(emb_x)
        out = torch.permute(out, (1, 0, 2))
        y = self.classifier(out) # B x S x E
        return y

In [72]:
# Bidirectional training
ner_model = NERModelElman(input_dim = 1024, hidden_dim = 256,
                          num_layers = 2, embedding_model = EMBD_MODEL, bi = True).to(device='cuda')

In [87]:
ner_model.rnn.load_state_dict(torch.load("C:\\main\\GitHub\\documentReviewSystem\\NERTraining\\models\\BI_ELMAN_RNN_LAYERS_2_HIDDEN_256_epoch#54.pth"))

<All keys matched successfully>

In [102]:
model_inf = ModelInference(VAL_PDF_DOC_PATH, 5, TOKENIZER)
model_inf

In [103]:
inf_result = model_inf.infer(ner_model, device = 'cuda')
inf_result

['и логистическая регрессии',
 'Полиноми',
 'вектор',
 ', гаус',
 '. Перцепт',
 'Наив',
 'ная байесов',
 'классификации',
 'и регрессии',
 'моделей регрессии',
 'Недообу',
 'переобу',
 'bias',
 'переобучени',
 'реляцион',
 'реляци']

### bidirectional GRU inference

In [26]:
@dataclass(frozen=True)
class TrainConfig:
    BATCH_SIZE: int = 6
    LR: float = 0.001
    BETAS: Tuple =  (0.9, 0.98)
    BIDIRECTIONAL: bool = False
train_config = TrainConfig()

In [41]:
# Bidirectional GRU training
class NERModelGRU(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int,
                 embedding_model: AutoTokenizer, num_layers: int = 2, num_tags: int = 3,
                 bi: bool = False):
        super().__init__()
        self.embedding_model = embedding_model
        self.classifier = nn.Linear(2*hidden_dim if bi else hidden_dim, num_tags)
        self.rnn = nn.GRU(input_size = input_dim, hidden_size = hidden_dim,
                           num_layers = num_layers, bidirectional = bi, batch_first = False)

    def forward(self, x: torch.tensor) -> torch.tensor:
        """x is tensor of shape BxS where B is batch size and S is sequence length of inputs"""
        with torch.no_grad():
            emb_x = self.embedding_model(x).last_hidden_state
        emb_x = torch.permute(emb_x, (1, 0, 2))
        out, _ = self.rnn(emb_x)
        out = torch.permute(out, (1, 0, 2))
        y = self.classifier(out) # B x S x E
        return y

In [44]:
ner_model = NERModelGRU(input_dim = 1024, hidden_dim = 512, num_layers = 1,
                        embedding_model = EMBD_MODEL, bi = True).to(device='cuda')
ner_model.rnn.load_state_dict(torch.load("C:\\main\\GitHub\\documentReviewSystem\\NERTraining\\models\\BI_GRU_LAYERS_1_HIDDEN_256_epoch#37.pth"))

<All keys matched successfully>

In [ ]:
model_inf = ModelInference(VAL_PDF_DOC_PATH, 5, TOKENIZER)

In [61]:
inf_result = model_inf.infer(ner_model, device = 'cuda')
inf_result

['MAE',
 ', RMSE,',
 'MSLE, M',
 '- accuracy',
 ', precis',
 ', PR',
 '. Не',
 'дообучение',
 'обу',
 'bias-vari',
 'и пере',
 'ционная форма данных',
 ', понятие',
 ') анализ',
 'в данных',
 ', воспроизводимость преобразован',
 'ия данных',
 '.']

### bidirectional LSTM inference

In [62]:
@dataclass(frozen=True)
class TrainConfig:
    BATCH_SIZE: int = 6
    LR: float = 0.001
    BETAS: Tuple =  (0.9, 0.98)
    BIDIRECTIONAL: bool = False
train_config = TrainConfig()

In [63]:
class NERModelLSTM(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int,
                 embedding_model: AutoTokenizer, num_layers: int = 2, num_tags: int = 3,
                 bi: bool = False):
        super().__init__()
        self.embedding_model = embedding_model
        self.classifier = nn.Linear(2*hidden_dim if bi else hidden_dim, num_tags)
        self.rnn = nn.LSTM(input_size = input_dim, hidden_size = hidden_dim,
                           num_layers = num_layers, bidirectional = bi, batch_first = False)

    def forward(self, x: torch.tensor) -> torch.tensor:
        """x is tensor of shape BxS where B is batch size and S is sequence length of inputs"""
        with torch.no_grad():
            emb_x = self.embedding_model(x).last_hidden_state
        emb_x = torch.permute(emb_x, (1, 0, 2)) 
        out, _ = self.rnn(emb_x)
        out = torch.permute(out, (1, 0, 2))
        y = self.classifier(out) # B x S x E
        return y

In [76]:
# bidirectional lstm training
ner_model = NERModelLSTM(input_dim = 1024, hidden_dim = 256,
                         num_layers = 1, embedding_model = EMBD_MODEL, bi = True).to(device='cuda')

In [77]:
ner_model.rnn.load_state_dict(torch.load("C:\\main\\GitHub\\documentReviewSystem\\NERTraining\\models\\BI_LSTM_LAYERS_1_HIDDEN_256_epoch#45.pth"))

<All keys matched successfully>

In [84]:
model_inf = ModelInference(VAL_PDF_DOC_PATH, 17, TOKENIZER)

In [85]:
inf_result = model_inf.infer(ner_model, device = 'cuda')
inf_result

['мате',
 'профессиональных задач',
 'ной предметной области при',
 'результаты модел',
 'Загруз',
 'датасе',
 'та.',
 'данных,',
 'к обучению об',
 'несколько моделей',
 '.']